# 🐭 マウスひっかき行動自動検出システム — Google Colab 版

本ノートブックは、**ボタンを押すだけ**でマウスのひっかき行動を自動検出できるシステムを Google Colab 上で動作させます。

### 動作フロー
```
実験動画 (.mp4/.mov) または SLEAP .h5 ファイル
       ↓
   姿勢推定 (YOLOv8-pose / SLEAP)
       ↓
   カラーID による個体識別
       ↓
   ひっかき行動の自動検出 & Bout 集計
       ↓
   Web ダッシュボードで可視化・手動補正・CSV エクスポート
```

### 推奨実行手順
1. ランタイム → ランタイムのタイプを変更 → **GPU (T4)** を選択
2. 各セルを上から順に実行 (`Shift+Enter`)
3. ステップ6 で発行される URL をクリックしてダッシュボードを開く

---


## ⚡ ステップ0: GPU 確認
T4 GPU が割り当てられているか確認します。GPU なしでも動作しますが、姿勢推定が遅くなります。


In [ ]:
import subprocess, sys, os

print('=' * 55)
print(' GPU / 環境チェック')
print('=' * 55)

# GPU 確認
try:
    result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
                             '--format=csv,noheader'], capture_output=True, text=True)
    if result.returncode == 0:
        print(f'✅ GPU 検出: {result.stdout.strip()}')
    else:
        print('⚠️  GPU が検出されませんでした。CPU モードで動作します。')
        print('   [ランタイム] → [ランタイムのタイプを変更] → GPU を選択することで高速化できます。')
except FileNotFoundError:
    print('⚠️  nvidia-smi が見つかりません。CPU モードで動作します。')

# Python バージョン確認
print(f'✅ Python: {sys.version.split()[0]}')

# Colab 環境確認
try:
    import google.colab
    print('✅ Google Colab 環境を検出しました')
    IN_COLAB = True
except ImportError:
    print('ℹ️  ローカル Jupyter 環境として動作します')
    IN_COLAB = False

print('=' * 55)

# グローバル変数として保存
LOCAL_PROJECT_PATH = '/content/mouse-scratch-dashboard'
print(f'プロジェクトパス: {LOCAL_PROJECT_PATH}')


---
## 🛠️ ステップ1: Google Drive の接続
実験動画や SLEAP データを Google Drive から直接読み込むため、Drive をマウントします。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive をマウントしました')


---
## 📂 ステップ2: プロジェクトフォルダの配置
Google Drive 上のプロジェクトフォルダを、Colab のローカル高速ディスク (`/content`) にコピーします。

> **`DRIVE_PROJECT_PATH`** を自分の Google Drive 上のパスに変更してください。


In [ ]:
import os, shutil

# ▼ ご自身の Google Drive 上のパスを指定してください
DRIVE_PROJECT_PATH = '/content/drive/MyDrive/mouse-scratch-dashboard'
LOCAL_PROJECT_PATH = '/content/mouse-scratch-dashboard'

if os.path.exists(DRIVE_PROJECT_PATH):
    if os.path.exists(LOCAL_PROJECT_PATH):
        shutil.rmtree(LOCAL_PROJECT_PATH)
    shutil.copytree(DRIVE_PROJECT_PATH, LOCAL_PROJECT_PATH)
    os.chdir(LOCAL_PROJECT_PATH)
    print(f'✅ プロジェクトをコピーしました: {LOCAL_PROJECT_PATH}')
    print(f'   作業ディレクトリ: {os.getcwd()}')
else:
    print(f'❌ フォルダが見つかりません: {DRIVE_PROJECT_PATH}')
    print('   Google Drive に mouse-scratch-dashboard フォルダをアップロードしてください。')
    raise FileNotFoundError(DRIVE_PROJECT_PATH)


---
## 📦 ステップ3: 依存ライブラリのインストール
FastAPI サーバー・YOLOv8・SLEAP・トンネリングツールをインストールします。

SLEAP のインストールは時間がかかる場合があります。失敗しても YOLOv8 で自動的に動作します。


In [ ]:
import subprocess, sys

def pip_install(*packages, quiet=True):
    cmd = [sys.executable, '-m', 'pip', 'install'] + list(packages)
    if quiet:
        cmd += ['-q', '--quiet']
    result = subprocess.run(cmd, capture_output=True, text=True)
    return result.returncode == 0, result.stderr

print('📦 [1/4] コアライブラリをインストール中...')
ok, err = pip_install('fastapi', 'uvicorn[standard]', 'pydantic>=2.0',
                      'opencv-python-headless', 'h5py', 'pandas', 'numpy<2')
print('  ✅ コアライブラリ' if ok else f'  ⚠️ エラー: {err[-300:]}')

print('📦 [2/4] YOLOv8 (ultralytics) をインストール中...')
ok, err = pip_install('ultralytics')
print('  ✅ ultralytics (YOLOv8)' if ok else f'  ⚠️ エラー: {err[-300:]}')

print('📦 [3/4] SLEAP をインストール中 (時間がかかります)...')
ok, err = pip_install('sleap', quiet=False)
if ok:
    try:
        import sleap
        print(f'  ✅ SLEAP v{sleap.__version__} インストール成功')
    except ImportError:
        print('  ⚠️ SLEAP インストール後も import できません。YOLOv8 を使用します。')
else:
    print('  ⚠️ SLEAP のインストールに失敗しました (TF 依存の競合が多い)。')
    print('     YOLOv8 エンジンを自動的に使用します。')

print('📦 [4/4] localtunnel (公開 URL 用) をインストール中...')
result = subprocess.run(['npm', 'install', '-g', 'localtunnel'],
                        capture_output=True, text=True)
print('  ✅ localtunnel' if result.returncode == 0 else f'  ⚠️ npm エラー: {result.stderr[-200:]}')

print('\n✅ インストール完了')


---
## 🤖 ステップ4: モデルの準備確認
YOLOv8 デフォルトモデル (`yolov8n-pose.pt`) の準備状況を確認します。
初回は自動ダウンロードされます（約 6MB）。


In [ ]:
import os, sys
os.chdir(LOCAL_PROJECT_PATH)

# setup_models.py を実行してモデルの準備状況を確認・表示
result = subprocess.run([sys.executable, 'setup_models.py'], capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print('[stderr]', result.stderr[-500:])


---
## 🌐 ステップ5: Colab インスタンスの IP アドレスを確認
localtunnel の認証に使用します。表示された IP アドレスを手元に控えてください。


In [ ]:
print('=' * 55)
print(' localtunnel 認証用 IP アドレス')
print('=' * 55)
!curl -s ipv4.icanhazip.com
print('\n↑ このIPアドレスをステップ7でコピー＆ペーストします')
print('=' * 55)


---
## 🚀 ステップ6: FastAPI バックエンドサーバーの起動
ダッシュボードのバックエンドをバックグラウンドで起動します。


In [ ]:
import subprocess, time, os, sys

os.chdir(LOCAL_PROJECT_PATH)

print('FastAPI サーバーを起動しています...')
server_process = subprocess.Popen(
    [sys.executable, 'app_server.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding='utf-8',
    errors='replace'
)

time.sleep(4)

# 起動ログを表示
print('--- サーバー起動ログ ---')
for _ in range(8):
    line = server_process.stdout.readline()
    if line:
        print(f'  {line.rstrip()}')
print('------------------------')

# 動作確認
import urllib.request
try:
    with urllib.request.urlopen('http://localhost:8000/', timeout=5) as r:
        print(f'\n✅ サーバー起動確認: HTTP {r.status}')
except Exception as e:
    print(f'\n⚠️  接続確認に失敗しました: {e}')
    print('   少し待ってから次のステップに進んでください。')


---
## 🔗 ステップ7: 外部公開 URL の生成
**方法 A**: localtunnel（推奨。IPアドレスの入力が必要）

**方法 B**: Cloudflare Tunnel（より安定。IPアドレス入力不要）

どちらか片方を実行してください。


In [ ]:
# ── 方法 A: localtunnel ──────────────────────────────────────────
import subprocess, time

print('🔗 [方法 A] localtunnel でトンネルを開いています...')
lt_process = subprocess.Popen(
    ['lt', '--port', '8000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(6)

url = None
for _ in range(10):
    line = lt_process.stdout.readline()
    if 'your url is' in line.lower():
        url = line.split('is:')[-1].strip()
        break

print('\n' + '=' * 65)
print('🌐 ダッシュボードへのアクセス手順 (localtunnel)')
print('=' * 65)
if url:
    print(f'\n  👉  {url}  👈\n')
else:
    print('  ⚠️  URL の取得に失敗しました。少し待ってから再実行してください。')
    print('  または「方法 B」(Cloudflare) セルを実行してください。')
print('手順:')
print('  1. 上のURLをクリック')
print('  2. 表示される入力欄にステップ5の IP アドレスを入力')
print('  3. "Click to Submit" をクリック → ダッシュボードが表示されます')
print('=' * 65)


In [ ]:
# ── 方法 B: Cloudflare Tunnel (IP 入力不要・より安定) ──────────
import subprocess, threading, time, re, os

# cloudflared のダウンロード (まだなければ)
cf_bin = '/usr/local/bin/cloudflared'
if not os.path.exists(cf_bin):
    print('📥 cloudflared をダウンロード中...')
    subprocess.run([
        'wget', '-q',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        '-O', cf_bin
    ], check=True)
    subprocess.run(['chmod', '+x', cf_bin], check=True)
    print('✅ cloudflared を準備しました')

url_found = []

def _read_cf(proc):
    pattern = re.compile(r'https://[a-z0-9\-]+\.trycloudflare\.com')
    for line in proc.stderr:
        m = pattern.search(line)
        if m:
            url_found.append(m.group())
            break

cf_proc = subprocess.Popen(
    [cf_bin, 'tunnel', '--url', 'http://localhost:8000'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.PIPE,
    text=True
)

t = threading.Thread(target=_read_cf, args=(cf_proc,), daemon=True)
t.start()
t.join(timeout=20)

print('\n' + '=' * 65)
print('🌐 ダッシュボードへのアクセス (Cloudflare Tunnel)')
print('=' * 65)
if url_found:
    print(f'\n  👉  {url_found[0]}  👈\n')
    print('  このURLをクリックするだけでダッシュボードが開きます。')
    print('  (IP アドレスの入力は不要です)')
else:
    print('  ⚠️  URL の取得に失敗しました。「方法 A」を試してください。')
print('=' * 65)


---
## 💡 ダッシュボードの使い方

### 動画を解析する
ダッシュボード左パネルの「解析の実行 (Run Analysis)」に動画パスを入力して実行します。

**Google Drive 上の動画の場合:**
```
/content/drive/MyDrive/mouse_videos/cage_01.mp4
```
(Colab 左パネルのファイルブラウザで右クリック →「パスをコピー」で取得できます)

**SLEAP .h5 ファイルを既にお持ちの場合:**
- エンジンに `sleap` を選択し、動画パスと H5 ファイルパスを指定してください
- またはノートブック下部の「ステップ9」で直接 pipeline.py を呼び出すこともできます

### 手動補正
- Canvas 上の関節点をドラッグして位置を修正
- 「Bulk Offset」でフレーム全体のオフセットを一括適用
- 「Swap ID」で個体の取り違えを即座に修正
- 右上「Export CSV」でデータをダウンロード

---


---
## 🔬 ステップ8 (応用): 複数モデルによる一括比較解析

同一動画に対して以下の手法で姿勢推定を実行し、ダッシュボードで精度比較を行います。

| 手法 | エンジン | モデル | キーポイント数 | 特徴 |
|------|----------|--------|---------------|------|
| 1 | DLC SuperAnimal | superanimal_topviewmouse (zero-shot) | 27 | 真上視点・マウス専用・追加学習不要 |
| 2 | DLC SuperAnimal | superanimal_quadruped (zero-shot) | 39 | 側面視点・**paws を含む**・最も解剖学的に正確 |
| 3 | SLEAP | Zenodo SuperAnimal-TopViewMouse-5K | 27 | 学術用大規模事前学習 |
| 4 | SLEAP | SLEAP 公式チュートリアル | テンプレート | 標準テンプレート |
| 5 | YOLOv8 | yolov8n-pose.pt (人間用ベース) | 17 (流用) | 軽量・高速・解剖学的精度は低い |
| 6 | YOLOv8 | Roboflow マウスデータで追加学習 | カスタム | 高速かつカスタム精度 |

> **DLC SuperAnimal は追加学習が不要で、本リポジトリで最も簡単かつ精度の高いマウス推定手段です (ステップ11 で詳説)。**


In [ ]:
# 比較検証用の追加パッケージ
print('📦 追加パッケージのインストール中...')
!pip install -q ultralytics roboflow h5py

# SLEAP (まだインストールされていない場合)
try:
    import sleap
    print(f'✅ SLEAP v{sleap.__version__} は既にインストール済みです')
except ImportError:
    print('📦 SLEAP をインストール中 (数分かかります)...')
    !pip install -q sleap
    try:
        import sleap
        print(f'✅ SLEAP v{sleap.__version__} インストール完了')
    except ImportError:
        print('⚠️ SLEAP のインストールに失敗しました。手法1・2はスキップされます。')


In [ ]:
# ── 手法1: Zenodo SuperAnimal-TopViewMouse-5K モデルのダウンロード ──
import os, urllib.request

os.chdir(LOCAL_PROJECT_PATH)
zenodo_url = 'https://zenodo.org/records/13757509/files/TopViewMouse5K_20240913.tar.gz?download=1'
tar_path   = '/content/TopViewMouse5K.tar.gz'
model_dir  = f'{LOCAL_PROJECT_PATH}/models/zenodo_mouse_5k'

os.makedirs(model_dir, exist_ok=True)

if not os.path.exists(tar_path):
    print('📥 Zenodo モデル (1.4 GB) をダウンロード中...')
    urllib.request.urlretrieve(zenodo_url, tar_path)
    print('✅ ダウンロード完了。展開中...')
    !tar -xzf {tar_path} -C {model_dir}
    print(f'✅ 手法1 モデルを配置しました: {model_dir}')
else:
    print(f'✅ 手法1 モデルは既にダウンロード済みです')


In [ ]:
# ── 手法2: SLEAP 公式テンプレートモデルのダウンロード ──
import os, urllib.request

tutorial_url  = 'https://github.com/talmolab/sleap/raw/main/tests/data/labels/labels.slp'
tutorial_dest = f'{LOCAL_PROJECT_PATH}/models/sleap_tutorial.slp'

if not os.path.exists(tutorial_dest):
    urllib.request.urlretrieve(tutorial_url, tutorial_dest)
    print(f'✅ 手法2 SLEAP テンプレートを配置しました: {tutorial_dest}')
else:
    print(f'✅ 手法2 モデルは既に配置済みです')


In [ ]:
# ── 手法4: Roboflow データセットで YOLOv8-pose を追加学習 ──
# ▼ Roboflow アカウントの API キーを入力してください (無料)
#   https://roboflow.com/ でサインアップ → 右上のアイコン → API Keys
ROBOFLOW_API_KEY = 'YOUR_ROBOFLOW_API_KEY'

if ROBOFLOW_API_KEY == 'YOUR_ROBOFLOW_API_KEY':
    print('⚠️  ROBOFLOW_API_KEY が未設定のため、手法4 (ファインチューニング) をスキップします。')
    print('   手法3 (YOLOv8 標準モデル) は引き続き使用可能です。')
else:
    from roboflow import Roboflow
    from ultralytics import YOLO
    import shutil

    rf      = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace('sl-h33qa').project('mouse_poseestimate')
    dataset = project.version(1).download('yolov8')
    print('✅ Roboflow データセットをダウンロードしました')

    print('🚀 YOLOv8-pose ファインチューニング (10 epochs) 開始...')
    model = YOLO('yolov8n-pose.pt')
    model.train(data=os.path.join(dataset.location, 'data.yaml'),
                epochs=10, imgsz=640, device=0,
                project='mouse_pose_yolo', name='finetuned')

    best_pt = 'mouse_pose_yolo/finetuned/weights/best.pt'
    dest_pt = f'{LOCAL_PROJECT_PATH}/models/yolov8_mouse_finetuned.pt'
    if os.path.exists(best_pt):
        shutil.copy2(best_pt, dest_pt)
        print(f'✅ 学習済みモデルを配置しました: {dest_pt}')


In [ ]:
# ── 同一動画に 4 モデルを一括適用 ──────────────────────────────
import os, sys

os.chdir(LOCAL_PROJECT_PATH)

# ▼ 解析対象の動画パスを指定してください
TARGET_VIDEO = f'{LOCAL_PROJECT_PATH}/1000.mov'

if not os.path.exists(TARGET_VIDEO):
    print(f'⚠️  動画ファイルが見つかりません: {TARGET_VIDEO}')
    print('   TARGET_VIDEO のパスを修正してから実行してください。')
else:
    zenodo_model   = f'{LOCAL_PROJECT_PATH}/models/zenodo_mouse_5k'
    tutorial_model = f'{LOCAL_PROJECT_PATH}/models/sleap_tutorial.slp'
    custom_model   = f'{LOCAL_PROJECT_PATH}/models/yolov8_mouse_finetuned.pt'

    tasks = [
        ('手法1: Zenodo SLEAP',  ['--engine', 'sleap', '--model_path', zenodo_model],
         'outputs/result_zenodo', os.path.exists(zenodo_model)),
        ('手法2: SLEAP チュートリアル', ['--engine', 'sleap', '--model_path', tutorial_model],
         'outputs/result_sleap_tutorial', os.path.exists(tutorial_model)),
        ('手法3: YOLOv8 標準',   ['--engine', 'yolov8', '--model_path', 'yolov8n-pose.pt'],
         'outputs/result_yolov8_standard', True),
        ('手法4: YOLOv8 カスタム', ['--engine', 'yolov8', '--model_path', custom_model],
         'outputs/result_yolov8_finetuned', os.path.exists(custom_model)),
    ]

    for i, (name, engine_args, out_dir, available) in enumerate(tasks, 1):
        if not available:
            print(f'  [{i}/4] {name}: モデルが見つからないためスキップ')
            continue
        print(f'\n🔥 [{i}/4] {name} で解析中...')
        cmd = [sys.executable, 'pipeline.py', '--video', TARGET_VIDEO,
               '--out_dir', out_dir] + engine_args
        result = subprocess.run(cmd, capture_output=True, text=True, encoding='utf-8')
        if result.returncode == 0:
            print(f'  ✅ 完了: {out_dir}')
        else:
            print(f'  ⚠️ エラー: {result.stdout[-500:]}')

    print('\n✨ 全モデルの一括解析が完了しました！')
    print('ダッシュボードの「比較解析結果の自動ロード」ドロップダウンから結果を切り替えて比較できます。')


---
## 💾 ステップ9: 解析結果を Google Drive にバックアップ


In [ ]:
import os, shutil

DRIVE_BACKUP_DIR = '/content/drive/MyDrive/mouse-scratch-dashboard/outputs'
local_outputs    = os.path.join(LOCAL_PROJECT_PATH, 'outputs')

if not os.path.exists(local_outputs):
    print('❌ outputs/ フォルダが見つかりません。先に解析を実行してください。')
else:
    os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
    copied = 0
    for root, dirs, files in os.walk(local_outputs):
        for f in files:
            if f.endswith('.csv'):
                src = os.path.join(root, f)
                rel = os.path.relpath(src, local_outputs)
                dst = os.path.join(DRIVE_BACKUP_DIR, rel)
                os.makedirs(os.path.dirname(dst), exist_ok=True)
                shutil.copy2(src, dst)
                print(f'  💾 {rel}')
                copied += 1
    if copied == 0:
        print('保存対象の CSV ファイルが見つかりません。')
    else:
        print(f'\n✅ {copied} 件の CSV ファイルを Google Drive に保存しました。')
        print(f'   保存先: {DRIVE_BACKUP_DIR}')


---
## 🔧 ステップ10 (応用): SLEAP .h5 ファイルを直接インポートして解析

すでに SLEAP で姿勢推定済みの `.h5` ファイルがある場合、そのデータを直接使って
特徴量抽出とひっかき行動検出だけを実行できます。


In [ ]:
import os, sys
os.chdir(LOCAL_PROJECT_PATH)

# ▼ 以下を自分の動画・H5 ファイルのパスに変更してください
VIDEO_PATH = '/content/drive/MyDrive/mouse_videos/cage_01.mp4'
SLEAP_H5   = '/content/drive/MyDrive/mouse_videos/cage_01.analysis.h5'
OUT_DIR    = 'outputs/from_sleap_h5'

if not os.path.exists(SLEAP_H5):
    print(f'⚠️  H5 ファイルが見つかりません: {SLEAP_H5}')
    print('   パスを修正してから実行してください。')
else:
    cmd = [
        sys.executable, 'pipeline.py',
        '--video', VIDEO_PATH,
        '--sleap', SLEAP_H5,
        '--out_dir', OUT_DIR
    ]
    print(f'▶ 実行: {" ".join(cmd)}')
    result = subprocess.run(cmd, capture_output=True, text=True, encoding='utf-8')
    print(result.stdout)
    if result.returncode != 0:
        print('[Error]', result.stderr[-1000:])


---
## 🐭 ステップ11 (推奨): DeepLabCut SuperAnimal による高精度マウス専用推論

**DeepLabCut Model Zoo** に公開されている **SuperAnimal** モデルは、研究グループ DLC Team によって学習された
マウス姿勢推定モデルで、**追加学習なし (zero-shot) でそのまま使えます**。

| モデル | キーポイント数 | 視点 | 含まれる解剖学的特徴 | 推奨用途 |
|--------|------------|------|----------------------|----------|
| `superanimal_topviewmouse` | 27 点 | **真上** | nose, ears, shoulders, hips, tail, mid-back | ケージを上から撮影した動画 |
| `superanimal_quadruped`    | 39 点 | **側面/斜め** | nose, ears, **paws (4 足全て)**, joints, tail | paws の解剖学的正確さが重要な解析 |

> **真上視点動画には TopView**, **paws が見える側面/斜め視点動画には Quadruped** を選択してください。
> TopView モデルは paw キーポイントを含まないため、`left_hind_paw` 等は `left_hip` の近似値となります。


In [ ]:
# DeepLabCut Model Zoo (SuperAnimal) のインストール
# DLC v3.0+ で zero-shot 推論 API (video_inference_superanimal) が利用可能
# 重い依存 (PyTorch / HRNet 等) を含むため数分かかります
import subprocess, sys

print('DeepLabCut[modelzoo,pytorch] をインストール中 (数分かかります)...')
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'deeplabcut[modelzoo,pytorch]>=3.0.0'],
    capture_output=True, text=True
)
if result.returncode == 0:
    try:
        import deeplabcut as dlc
        ver = getattr(dlc, '__version__', '?')
        has_sa = hasattr(dlc, 'video_inference_superanimal')
        print(f'OK DeepLabCut v{ver} インストール成功')
        if has_sa:
            print('OK SuperAnimal API (zero-shot 推論) が利用可能です')
        else:
            print('WARN: SuperAnimal API が見つかりません。DLC が v3.0+ か確認してください。')
    except ImportError as e:
        print(f'WARN: DLC をインポートできません: {e}')
else:
    print(f'WARN: インストール失敗 (TF/PyTorch 依存の競合): {result.stderr[-500:]}')


In [ ]:
# DLC SuperAnimal-TopViewMouse で解析する (真上視点の動画用)
import os, sys, subprocess

os.chdir(LOCAL_PROJECT_PATH)

# 真上視点で撮影された動画ファイルのパスを指定
TARGET_VIDEO_TOPVIEW = f'{LOCAL_PROJECT_PATH}/1000.mov'

if not os.path.exists(TARGET_VIDEO_TOPVIEW):
    print(f'WARN: 動画ファイルが見つかりません: {TARGET_VIDEO_TOPVIEW}')
    print('   TARGET_VIDEO_TOPVIEW のパスを修正してから実行してください。')
else:
    cmd = [
        sys.executable, 'pipeline.py',
        '--video', TARGET_VIDEO_TOPVIEW,
        '--engine', 'dlc',
        '--dlc-superanimal', 'superanimal_topviewmouse',
        '--out_dir', 'outputs/result_dlc_topview',
        # ご自身のカメラで実測した値に置き換えてください
        '--px-to-mm', '0.25',
    ]
    print('実行:', ' '.join(cmd))
    print('   初回は SuperAnimal モデル (~数百MB) のダウンロードに時間がかかります。')
    result = subprocess.run(cmd, capture_output=True, text=True, encoding='utf-8')
    print(result.stdout[-3000:])
    if result.returncode != 0:
        print('[Error]', result.stderr[-1500:])
    else:
        print('\nOK DLC TopView 解析完了: outputs/result_dlc_topview/')


In [ ]:
# DLC SuperAnimal-Quadruped で解析する (側面/斜め視点・paws 含む)
import os, sys, subprocess

os.chdir(LOCAL_PROJECT_PATH)

# 側面または斜め視点で撮影された動画ファイルのパスを指定
TARGET_VIDEO_SIDE = f'{LOCAL_PROJECT_PATH}/1000.mov'

if not os.path.exists(TARGET_VIDEO_SIDE):
    print(f'WARN: 動画ファイルが見つかりません: {TARGET_VIDEO_SIDE}')
else:
    cmd = [
        sys.executable, 'pipeline.py',
        '--video', TARGET_VIDEO_SIDE,
        '--engine', 'dlc',
        '--dlc-superanimal', 'superanimal_quadruped',
        '--out_dir', 'outputs/result_dlc_quadruped',
        '--px-to-mm', '0.25',
    ]
    print('実行:', ' '.join(cmd))
    print('   Quadruped モデルは paws を直接含むため、ひっかき検出には解剖学的に最適です。')
    result = subprocess.run(cmd, capture_output=True, text=True, encoding='utf-8')
    print(result.stdout[-3000:])
    if result.returncode != 0:
        print('[Error]', result.stderr[-1500:])
    else:
        print('\nOK DLC Quadruped 解析完了: outputs/result_dlc_quadruped/')


In [ ]:
# (応用) 全エンジン (YOLOv8 / DLC-TopView / DLC-Quadruped) で
#        同一動画を解析し、ダッシュボードのドロップダウンから精度比較する
import os, sys, subprocess

os.chdir(LOCAL_PROJECT_PATH)

TARGET_VIDEO = f'{LOCAL_PROJECT_PATH}/1000.mov'

if not os.path.exists(TARGET_VIDEO):
    print(f'WARN: 動画ファイルが見つかりません: {TARGET_VIDEO}')
else:
    tasks = [
        ('YOLOv8 (人間COCO流用)',  ['--engine', 'yolov8'],
         'outputs/cmp_yolov8'),
        ('DLC SuperAnimal-TopView',
         ['--engine', 'dlc', '--dlc-superanimal', 'superanimal_topviewmouse'],
         'outputs/cmp_dlc_topview'),
        ('DLC SuperAnimal-Quadruped',
         ['--engine', 'dlc', '--dlc-superanimal', 'superanimal_quadruped'],
         'outputs/cmp_dlc_quadruped'),
    ]
    for i, (name, engine_args, out_dir) in enumerate(tasks, 1):
        print(f'\n[{i}/{len(tasks)}] {name} で解析中...')
        cmd = [sys.executable, 'pipeline.py', '--video', TARGET_VIDEO,
               '--out_dir', out_dir, '--px-to-mm', '0.25'] + engine_args
        result = subprocess.run(cmd, capture_output=True, text=True, encoding='utf-8')
        if result.returncode == 0:
            print(f'  OK 完了: {out_dir}')
        else:
            print(f'  WARN エラー: {(result.stdout or "")[-400:]}')

    print('\n全エンジン比較解析完了。ダッシュボードのドロップダウンから結果を切り替えて比較してください。')
